# 4.8 Function Calling 原理实战

对应笔记 **4.7 MCP → 4.7.1 从 Function calling 到 MCP → Function calling**。

本 notebook **手写一遍原生 Function calling 流程**,不依赖 SDK 内置的 `tools`/`tool_choice` 参数,完全按笔记里的机制来,步骤与笔记一一对应:

| # | 笔记「Function calling」 | 本 notebook |
|---|------------------------|-------------|
| 1 | **函数定义**——JSON Schema 描述工具 | §1 定义 `get_current_weather` |
| 2 | **工具调用判定**——输出 `<tool_call>` 标记 | §3 第一轮模型调用 |
| 3 | **生成函数调用**——输出 `name` + `arguments` | §3 同轮输出的 JSON |
| 4 | **工具执行**——输出解析 + 函数运行 | §4 正则解析 → §5 映射执行 |
| 5 | **模型输出**——`<tool_response>` 回传 + 最终答案 | §6 封装回传 → §7 再次调用 |

> 用真实 API(阿里云 DashScope 的 Qwen)演示,这样你能看清模型真实的输出。

## 0. 环境准备

```bash
pip install openai
```

把 `DASHSCOPE_API_KEY` 配到环境变量中,或直接填在下面单元格里(注意不要外传)。

In [1]:
import os
from openai import OpenAI

# ==================== 配置 ====================
# 笔记主题是 Qwen,这里走阿里云 DashScope 的 OpenAI 兼容接口
client = OpenAI(
    api_key=os.environ.get("DASHSCOPE_API_KEY") or "sk-xxxxxxxx",  # 填入你的 key
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
MODEL = "qwen-plus"  # 模型也要能稳定输出 <tool_call> 标签,用带工具能力的即可

# ==================== 提示词 ====================
SYSTEM_PROMPT = """你是一个会调用外部工具的智能体。
当你需要查询实时数据(比如天气)时,必须严格按照以下格式输出工具调用:
<tool_call>
{"name": "函数名", "arguments": {"参数1": "值"}}
</tool_call>
函数定义如下:
"""
print("✅ 客户端就绪,模型:", MODEL)

✅ 客户端就绪,模型: qwen-plus


## 1. 函数定义(JSON Schema)

与笔记中的定义一致:`get_current_weather(location)`,参数是 `location`(城市或县区)。
把它格式化成提示词,拼进系统信息。

In [2]:
import json

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "当你想查询指定城市的天气时非常有用。",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "城市或县区,比如北京市、杭州市、余杭区等。",
                    }
                },
                "required": ["location"],
            },
        },
    }
]

# 把 JSON Schema 序列化成文本,拼到系统提示词里
schema_text = json.dumps(TOOLS, ensure_ascii=False, indent=4)
system_content = SYSTEM_PROMPT + schema_text
print(system_content)

你是一个会调用外部工具的智能体。
当你需要查询实时数据(比如天气)时,必须严格按照以下格式输出工具调用:
<tool_call>
{"name": "函数名", "arguments": {"参数1": "值"}}
</tool_call>
函数定义如下:
[
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "当你想查询指定城市的天气时非常有用。",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "城市或县区,比如北京市、杭州市、余杭区等。"
                    }
                },
                "required": [
                    "location"
                ]
            }
        }
    }
]


## 2. 工具实现(真实函数)

调度器要用函数名映射到真实函数。定义一个本地天气查询函数,并维护一个注册表 `FUNCTIONS`。

In [3]:
# 模拟数据源:真实工程中这里会是天气 API 的调用
WEATHER_DB = {
    "北京":  {"weather": "晴", "temperature": 32, "wind": "北风2级"},
    "杭州":  {"weather": "小雨", "temperature": 27, "wind": "东风3级"},
    "余杭区": {"weather": "阴", "temperature": 26, "wind": "微风"},
    "上海":  {"weather": "多云", "temperature": 30, "wind": "南风3级"},
}

def get_current_weather(location: str) -> dict:
    """查询指定城市的当前天气。"""
    # 规范化:模型常把用户说的"北京"输出成"北京市",数据库里存的是"北京"
    key = location.rstrip("市")
    if key in WEATHER_DB:
        return {"city": location, **WEATHER_DB[key]}
    return {"city": location, "error": f"未收录 {location} 的天气数据"}

# 函数名 -> 真实函数 的映射(注册表)
FUNCTIONS = {
    "get_current_weather": get_current_weather,
}

# 验证映射正确
print(get_current_weather("杭州市"))
print("✅ 已注册函数:", list(FUNCTIONS.keys()))

{'city': '杭州市', 'weather': '小雨', 'temperature': 27, 'wind': '东风3级'}
✅ 已注册函数: ['get_current_weather']


## 3. 工具调用判定 & 生成函数调用

对应笔记的「工具调用判定」与「生成函数调用」两节,这两步实际发生在**同一次模型调用**里:

- **工具调用判定**:模型在生成回答时判断用户需求是否超出自身知识范围,若需调用工具便输出特殊标记 `<tool_call>` 表明需要调用函数;
- **生成函数调用**:在 `<tool_call>` 标记后,模型输出**严格遵循预先提供 JSON Schema** 的调用请求,包含 `name`(要调用的函数名)和 `arguments`(参数)。

下面用流式方式发起第一轮调用,观察模型既输出判定标记、又生成结构化调用。

In [4]:
messages = [
    {"role": "system", "content": system_content},
    {"role": "user", "content": "北京今天天气怎么样?"},
]

# 第一轮:模型决定是否调用工具(流式,首 token 约 0.5s 即可见)
print("===== 第一轮模型输出 =====\n")
resp = client.chat.completions.create(model=MODEL, messages=messages, stream=True)
collected = []
for chunk in resp:
    delta = chunk.choices[0].delta.content or ""
    collected.append(delta)
    print(delta, end="", flush=True)
first_answer = "".join(collected)
print("\n==========================")
messages.append({"role": "assistant", "content": first_answer})

===== 第一轮模型输出 =====

<tool_call>
{"name": "get_current_weather", "arguments": {"location": "北京市"}}
</tool_call>


## 4. 工具执行(一)输出解析

对应笔记「工具执行」的第一步:**通过正则表达式匹配所有 JSON 串,并反序列化为 Python 对象**。

In [5]:
import re

def parse_tool_calls(text: str) -> list[dict]:
    """用正则解析出所有 <tool_call>...</tool_call> 里的 JSON 对象。"""
    pattern = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.DOTALL)
    calls = []
    for match in pattern.finditer(text):
        try:
            calls.append(json.loads(match.group(1)))
        except json.JSONDecodeError as e:
            print("⚠️ JSON 解析失败:", e)
    return calls

tool_calls = parse_tool_calls(first_answer)
print("解析出的工具调用:", tool_calls)
assert tool_calls, "模型没有输出 <tool_call> 标记,请检查 key / 模型是否支持工具调用"

解析出的工具调用: [{'name': 'get_current_weather', 'arguments': {'location': '北京市'}}]


## 5. 工具执行(二)函数运行

对应笔记「工具执行」的第二步:**根据函数名映射到对应的函数,并将 `arguments` 中的字段作为参数,调用对应的函数**。

In [6]:
results = []
for call in tool_calls:
    fn = FUNCTIONS[call["name"]]          # 根据函数名映射到真实函数
    result = fn(**call["arguments"])      # arguments 作为参数传入
    results.append(result)
    print(f"→ 调用 {call['name']}{call['arguments']} 返回:{result}")

→ 调用 get_current_weather{'location': '北京市'} 返回:{'city': '北京市', 'weather': '晴', 'temperature': 32, 'wind': '北风2级'}


## 6. 模型输出(一)封装工具响应

对应笔记「模型输出」的开头:**工具执行结果被封装成 `<tool_response>...</tool_response>` 消息,追加到对话历史中,供模型下一轮生成时参考**。

注意:Qwen 会将工具响应**以 `user` 角色插入对话**(而不是 tool 角色),希望模型把工具响应作为**新的信息源**,基于此再回答。

In [7]:
tool_response = "\n".join(json.dumps(r, ensure_ascii=False) for r in results)
messages.append(
    {
        "role": "user",
        "content": f"<tool_response>\n{tool_response}\n</tool_response>",
    }
)

print("===== 当前对话历史(已追加 tool_response) =====")
for m in messages:
    print(f"[{m['role']}]")
    print(m["content"])
    print("-" * 40)

===== 当前对话历史(已追加 tool_response) =====
[system]
你是一个会调用外部工具的智能体。
当你需要查询实时数据(比如天气)时,必须严格按照以下格式输出工具调用:
<tool_call>
{"name": "函数名", "arguments": {"参数1": "值"}}
</tool_call>
函数定义如下:
[
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "当你想查询指定城市的天气时非常有用。",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "城市或县区,比如北京市、杭州市、余杭区等。"
                    }
                },
                "required": [
                    "location"
                ]
            }
        }
    }
]
----------------------------------------
[user]
北京今天天气怎么样?
----------------------------------------
[assistant]
<tool_call>
{"name": "get_current_weather", "arguments": {"location": "北京市"}}
</tool_call>
----------------------------------------
[user]
<tool_response>
{"city": "北京市", "weath

## 7. 模型输出(二)生成最终答案

对应笔记「模型输出」的结尾:**再次调用大模型读取 `<tool_response>` 中的信息,生成准确的最终答案**。

In [8]:
# 第二轮:基于工具结果生成最终答案(流式)
print("===== 最终答案 =====\n")
resp = client.chat.completions.create(model=MODEL, messages=messages, stream=True)
collected = []
for chunk in resp:
    delta = chunk.choices[0].delta.content or ""
    collected.append(delta)
    print(delta, end="", flush=True)
final_answer = "".join(collected)
print("\n=====================")

===== 最终答案 =====

北京今天天气晴，气温32℃，风向为北风，风力2级。


## 8. 封装成完整循环(Agent 的最小形态)

上面 1~7 步串起来,就是一个 Agent 的最小闭环:用户提问 → 模型判工具 → 调度器执行 → 结果回传 → 模型作答。

封装成一个函数,再提问几个不同城市,验证流程可复用。

In [9]:
def chat_stream(messages: list) -> str:
    """封装一次流式对话:边生成边打印,返回完整文本。"""
    resp = client.chat.completions.create(model=MODEL, messages=messages, stream=True)
    collected = []
    for chunk in resp:
        delta = chunk.choices[0].delta.content or ""
        collected.append(delta)
        print(delta, end="", flush=True)
    print()  # 换行
    return "".join(collected)


def ask(question: str, verbose: bool = True) -> str:
    """一个完整的 Function calling 会话:问一句,返回最终答案。"""
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": question},
    ]

    # 1) 工具调用判定(流式)
    answer = chat_stream(messages)
    messages.append({"role": "assistant", "content": answer})

    # 2) 输出解析
    calls = parse_tool_calls(answer)
    if not calls:
        return answer  # 模型认为不需要工具,直接回答

    # 3) 工具执行
    results = []
    for call in calls:
        results.append(FUNCTIONS[call["name"]](**call["arguments"]))

    # 4) 模型输出:封装 tool_response,以 user 角色回传
    messages.append(
        {
            "role": "user",
            "content": "<tool_response>\n"
            + "\n".join(json.dumps(r, ensure_ascii=False) for r in results)
            + "\n</tool_response>",
        }
    )

    if verbose:
        print(f"🤖 判定的工具调用: {calls}")
        print(f"🛠️  工具返回: {results}")

    # 5) 基于工具结果生成最终答案(流式)
    return chat_stream(messages)


print(ask("上海今天天气怎么样?"))

<tool_call>
{"name": "get_current_weather", "arguments": {"location": "上海市"}}
</tool_call>
🤖 判定的工具调用: [{'name': 'get_current_weather', 'arguments': {'location': '上海市'}}]
🛠️  工具返回: [{'city': '上海市', 'weather': '多云', 'temperature': 30, 'wind': '南风3级'}]
上海今天天气为多云，气温30℃，风向为南风，风力3级。
上海今天天气为多云，气温30℃，风向为南风，风力3级。


In [10]:
print("Q:", "余杭区天气如何?")
print("A:", ask("余杭区天气如何?"))

Q: 余杭区天气如何?
<tool_call>
{"name": "get_current_weather", "arguments": {"location": "余杭区"}}
</tool_call>
🤖 判定的工具调用: [{'name': 'get_current_weather', 'arguments': {'location': '余杭区'}}]
🛠️  工具返回: [{'city': '余杭区', 'weather': '阴', 'temperature': 26, 'wind': '微风'}]
余杭区目前的天气是阴天，气温为26℃，风力为微风。
A: 余杭区目前的天气是阴天，气温为26℃，风力为微风。


In [11]:
print("Q:", "1+1 等于几?")
print("A:", ask("1+1 等于几?"))  # 不需要工具,模型直接回答

Q: 1+1 等于几?
1+1 等于 2。
A: 1+1 等于 2。


## 📊 总结

### 与笔记「Function calling」逐段对齐

| 笔记中的机制 | 本 notebook 的实现 | 对应代码 |
|------------|------------------|---------|
| **函数定义**:JSON Schema 描述函数,放进系统信息 | `TOOLS` + `SYSTEM_PROMPT` 拼接成 `system_content` | §1 cell 4 |
| **工具调用判定**:模型判断超出知识范围,输出 `<tool_call>` | 第一轮 `chat.completions.create` 的流式输出 | §3 cell 8 |
| **生成函数调用**:JSON Schema 严格约束,输出 `name` + `arguments` | `<tool_call>` 标签内的 `{"name":..., "arguments":{...}}` | §3 cell 8 |
| **工具执行·输出解析**:正则匹配所有 JSON 串并反序列化 | `parse_tool_calls()`(regex + `json.loads`) | §4 cell 10 |
| **工具执行·函数运行**:函数名映射 + `arguments` 传参 | `FUNCTIONS[name](**arguments)` | §5 cell 12 |
| **模型输出·封装响应**:`<tool_response>` 以 user 角色追加回对话 | 追加 `messages` 的 `user` 消息 | §6 cell 14 |
| **模型输出·最终答案**:再次调用模型读取响应并作答 | 第二轮调用(流式) | §7 cell 16 |

### 一个补充:笔记未提、实战会遇到的问题

- **地名规范化**:模型把"北京"输出成"北京市",而数据库 key 是"北京",需要 `rstrip("市")` 对齐。真实工具接入时这种"模型输出 ≠ 数据源格式"的差异很常见。
- **本 notebook 的"生成函数调用"用纯提示词约束**;真实工程(如 MCP)走 SDK 的 `tools=[...]` / `tool_choice` 结构化参数,由框架直接返回 `tool_calls` 字段,避免依赖模型"恰好"记得标签格式,鲁棒性更高。
- **MCP** 进一步把"工具注册表"标准化成协议:多个 Agent / 客户端共享同一套工具注册与调用协议,`FUNCTIONS` 硬编码映射对应到 MCP 就是 client 请求 server 的工具列表、再按 `name` 执行。

> 提示:把 `MODEL` 换成 `qwen-turbo` 或 `qwen-max` 都能跑;真实工程中 `WEATHER_DB` 会替换成真实的天气 API。